# 05 — LSTM Classifier
**Project:** IDS-KMUTT — AI-Based Intrusion Detection System  
**Dataset:** CICIDS2017 — post-SMOTE balanced version (4,542,640 rows)  
**Author:** Darren Touopi  
**Date:** 2026  

**Goal:** Train and evaluate a Long Short-Term Memory (LSTM) neural network on the CICIDS2017 dataset.  
The LSTM captures temporal dependencies between consecutive network flows using a sliding window of 10 flows.  
Compare results against Random Forest (Notebook 03) and XGBoost (Notebook 04).

**References:**  
- Hochreiter, S. & Schmidhuber, J. (1997). Long Short-Term Memory. Neural Computation, 9(8), 1735–1780.  
- Ferrag, M. A. et al. (2020). Deep Learning for Cyber Security Intrusion Detection. J. Information Security.

**Sequence length:** 10 consecutive flows (Ferrag et al. 2020)  
**GPU:** NVIDIA RTX 4090 (24GB VRAM) — gpu4090 partition

---
## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import joblib
import warnings
warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_curve, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
SEQUENCE_LEN = 10
BATCH_SIZE   = 512
EPOCHS       = 30
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {len(gpus)}")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU                : {gpus[0].name}")
else:
    print("No GPU — running on CPU (slower)")

---
## 2. Load dataset

In [ ]:
# Paths
SMOTE_PATH = "../data/processed/cicids2017_cleaned.csv"
MODEL_DIR  = "../models/"
FIGURE_DIR = "../figures/"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

print("Loading SMOTE-balanced dataset...")
t0 = time.time()
df = pd.read_csv(SMOTE_PATH)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

---
## 3. Prepare features and labels

In [ ]:
LABEL_COLS = ['Label_binary']
X = df.drop(columns=LABEL_COLS).values.astype(np.float32)
y = df['Label_binary'].values.astype(np.float32)
feature_names = df.drop(columns=LABEL_COLS).columns.tolist()

print(f"Features : {X.shape[1]}")
print(f"Class distribution:")
print(f"  BENIGN : {int((y==0).sum()):,}")
print(f"  ATTACK : {int((y==1).sum()):,}")

---
## 4. Normalize features

LSTM is sensitive to feature scale — StandardScaler normalizes each feature to mean=0, std=1.  
RF and XGBoost don't need normalization (tree-based), but LSTM does.

In [ ]:
print("Normalizing features with StandardScaler...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler — needed for inference in production
scaler_path = os.path.join(MODEL_DIR, 'lstm_scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"Scaler saved: {scaler_path}")
print(f"\nFeature mean (first 3): {scaler.mean_[:3].round(4)}")
print(f"Feature std  (first 3): {scaler.scale_[:3].round(4)}")

---
## 5. Create sequences (sliding window)

The LSTM processes sequences of consecutive network flows.  
We use a **sliding window of 10 flows** — each sequence = 10 consecutive rows.  
The label of each sequence = label of the last flow in the window.

**Reference:** Ferrag et al. (2020) use a window of 10 for CICIDS2017.

> ⚠️ **Memory note:** Creating sequences for 4.5M rows requires ~100GB RAM.  
> We subsample to 1M rows before sequence creation to stay within GPU node memory.

In [ ]:
SAMPLE_SIZE = 1_000_000

if len(X_scaled) > SAMPLE_SIZE:
    np.random.seed(RANDOM_STATE)
    idx = np.random.choice(len(X_scaled), SAMPLE_SIZE, replace=False)
    idx = np.sort(idx)  # Keep temporal order
    X_seq_data = X_scaled[idx]
    y_seq_data = y[idx]
    print(f"Subsampled to {SAMPLE_SIZE:,} rows (temporal order preserved)")
else:
    X_seq_data = X_scaled
    y_seq_data = y

print(f"Creating sequences (window={SEQUENCE_LEN})...")
t0 = time.time()

n_sequences = len(X_seq_data) - SEQUENCE_LEN
X_sequences = np.zeros((n_sequences, SEQUENCE_LEN, X_seq_data.shape[1]), dtype=np.float32)
y_sequences = np.zeros(n_sequences, dtype=np.float32)

for i in range(n_sequences):
    X_sequences[i] = X_seq_data[i:i + SEQUENCE_LEN]
    y_sequences[i] = y_seq_data[i + SEQUENCE_LEN - 1]

print(f"Done in {time.time()-t0:.1f}s")
print(f"Sequences shape : {X_sequences.shape}")
print(f"Memory usage    : {X_sequences.nbytes / 1e9:.2f} GB")
print(f"Label dist      : BENIGN={int((y_sequences==0).sum()):,} | ATTACK={int((y_sequences==1).sum()):,}")

---
## 6. Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_sequences,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_sequences
)

print(f"Train : {X_train.shape[0]:,} sequences ({X_train.shape}")
print(f"Test  : {X_test.shape[0]:,} sequences ({X_test.shape}")

---
## 7. Build LSTM model

Architecture inspired by Ferrag et al. (2020) — two stacked LSTM layers  
with BatchNormalization and Dropout for regularization.

In [ ]:
model = Sequential([
    # First LSTM layer — captures short-term temporal patterns
    LSTM(128, input_shape=(SEQUENCE_LEN, X.shape[1]),
         return_sequences=True, name='lstm_1'),
    BatchNormalization(),
    Dropout(0.3),

    # Second LSTM layer — captures longer-term dependencies
    LSTM(64, return_sequences=False, name='lstm_2'),
    BatchNormalization(),
    Dropout(0.3),

    # Dense classification head
    Dense(32, activation='relu', name='dense_1'),
    Dropout(0.2),
    Dense(1, activation='sigmoid', name='output')
], name='IDS_LSTM')

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

---
## 8. Train LSTM

**Callbacks:**
- **EarlyStopping** — stops training if val_loss doesn't improve for 5 epochs
- **ModelCheckpoint** — saves the best model (highest val_accuracy)
- **ReduceLROnPlateau** — halves learning rate if val_loss stagnates for 3 epochs

> 💡 **On RTX 4090:** ~2–5 min per epoch with batch_size=512 → total ~20–30 min  
> **On CPU:** ~30–60 min per epoch → not recommended

In [ ]:
checkpoint_path = os.path.join(MODEL_DIR, 'lstm_checkpoint.keras')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print(f"Training LSTM — max {EPOCHS} epochs, batch_size={BATCH_SIZE}...")
t0 = time.time()

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

t_train = time.time() - t0
print(f"\nTraining completed in {t_train/60:.1f} min")
print(f"Epochs run: {len(history.history['loss'])}")

---
## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='orange')
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='steelblue')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='orange')
axes[1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'lstm_training_curves.png'), dpi=150)
plt.show()

---
## 10. Evaluate on test set

In [ ]:
t0 = time.time()
y_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
t_infer = time.time() - t0

acc  = accuracy_score(y_test, y_pred)
f1m  = f1_score(y_test, y_pred, average='macro')
f1b  = f1_score(y_test, y_pred, average='binary')
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
fpr  = fp / (fp + tn)

print("=" * 50)
print("  LSTM — TEST SET RESULTS")
print("=" * 50)
print(f"  Accuracy         : {acc:.4f}")
print(f"  F1 (macro)       : {f1m:.4f}")
print(f"  F1 (binary)      : {f1b:.4f}")
print(f"  Precision        : {prec:.4f}")
print(f"  Recall (TPR)     : {rec:.4f}")
print(f"  ROC-AUC          : {auc:.4f}")
print(f"  FPR              : {fpr:.4f}")
print(f"  TP={tp:,}  FP={fp:,}  TN={tn:,}  FN={fn:,}")
print(f"  Inference time   : {t_infer:.2f}s ({len(X_test):,} sequences)")
print("=" * 50)
print()
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'ATTACK']))

---
## 11. Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['BENIGN', 'ATTACK'])
disp.plot(ax=ax, colorbar=True, cmap='Purples')
ax.set_title('LSTM — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'lstm_confusion_matrix.png'), dpi=150)
plt.show()
print(f"FPR: {fpr:.4f} — {fp:,} benign flows incorrectly flagged as attack")

---
## 12. ROC Curve

In [ ]:
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_curve, tpr_curve, color='purple', lw=2,
        label=f'LSTM (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve — LSTM', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'lstm_roc_curve.png'), dpi=150)
plt.show()

---
## 13. Threshold tuning

In [ ]:
thresholds_range = np.arange(0.1, 0.95, 0.05)
results = []

for thresh in thresholds_range:
    y_pred_t = (y_prob >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    results.append({
        'threshold': round(thresh, 2),
        'f1'       : round(f1_score(y_test, y_pred_t, average='binary'), 4),
        'precision': round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        'recall'   : round(recall_score(y_test, y_pred_t, zero_division=0), 4),
        'fpr'      : round(fp_t / (fp_t + tn_t), 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

candidates = results_df[results_df['recall'] >= 0.95]
if len(candidates) > 0:
    best_thresh_row = candidates.loc[candidates['fpr'].idxmin()]
    BEST_THRESHOLD = best_thresh_row['threshold']
    print(f"\n✅ Recommended threshold: {BEST_THRESHOLD}")
    print(f"   Recall: {best_thresh_row['recall']}  |  FPR: {best_thresh_row['fpr']}  |  F1: {best_thresh_row['f1']}")
else:
    BEST_THRESHOLD = 0.5

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(results_df['threshold'], results_df['f1'],        label='F1',        marker='o', color='purple')
ax.plot(results_df['threshold'], results_df['recall'],    label='Recall',    marker='s', color='green')
ax.plot(results_df['threshold'], results_df['precision'], label='Precision', marker='^', color='orange')
ax.plot(results_df['threshold'], results_df['fpr'],       label='FPR',       marker='x', color='red', linestyle='--')
ax.axvline(BEST_THRESHOLD, color='gray', linestyle=':', label=f'Best threshold = {BEST_THRESHOLD}')
ax.set_xlabel('Classification Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Tuning — LSTM', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, 'lstm_threshold_tuning.png'), dpi=150)
plt.show()

---
## 14. RF vs XGBoost vs LSTM comparison

In [ ]:
rf_path  = os.path.join(MODEL_DIR, 'rf_metrics.csv')
xgb_path = os.path.join(MODEL_DIR, 'xgb_metrics.csv')

if os.path.exists(rf_path) and os.path.exists(xgb_path):
    rf_m  = pd.read_csv(rf_path).iloc[0]
    xgb_m = pd.read_csv(xgb_path).iloc[0]

    comparison = pd.DataFrame({
        'Metric'       : ['Accuracy', 'F1 (macro)', 'F1 (binary)', 'Precision', 'Recall', 'ROC-AUC', 'FPR'],
        'Random Forest': [rf_m['accuracy'], rf_m['f1_macro'], rf_m['f1_binary'],
                          rf_m['precision'], rf_m['recall'], rf_m['roc_auc'], rf_m['fpr']],
        'XGBoost'      : [xgb_m['accuracy'], xgb_m['f1_macro'], xgb_m['f1_binary'],
                          xgb_m['precision'], xgb_m['recall'], xgb_m['roc_auc'], xgb_m['fpr']],
        'LSTM'         : [round(acc, 4), round(f1m, 4), round(f1b, 4),
                          round(prec, 4), round(rec, 4), round(auc, 4), round(fpr, 4)]
    })

    print("\n" + "=" * 65)
    print("  RF vs XGBoost vs LSTM — Head to Head")
    print("=" * 65)
    print(comparison.to_string(index=False))
    print("=" * 65)
else:
    print("RF or XGBoost metrics not found — run Notebooks 03 and 04 first.")

---
## 15. Save model and metrics

In [ ]:
# Save model
model_path = os.path.join(MODEL_DIR, 'lstm_binary_best.keras')
model.save(model_path)
print(f"Model saved      : {model_path}")

# Save metrics
lstm_metrics = {
    'model'         : 'LSTM',
    'sequence_len'  : SEQUENCE_LEN,
    'accuracy'      : round(acc, 4),
    'f1_macro'      : round(f1m, 4),
    'f1_binary'     : round(f1b, 4),
    'precision'     : round(prec, 4),
    'recall'        : round(rec, 4),
    'roc_auc'       : round(auc, 4),
    'fpr'           : round(fpr, 4),
    'tp'            : int(tp),
    'fp'            : int(fp),
    'tn'            : int(tn),
    'fn'            : int(fn),
    'best_threshold': BEST_THRESHOLD,
    'train_time_s'  : round(t_train, 1),
    'infer_time_s'  : round(t_infer, 2),
    'epochs_run'    : len(history.history['loss']),
    'batch_size'    : BATCH_SIZE,
}
metrics_path = os.path.join(MODEL_DIR, 'lstm_metrics.csv')
pd.DataFrame([lstm_metrics]).to_csv(metrics_path, index=False)
print(f"Metrics saved    : {metrics_path}")

print(f"\n✅ Notebook 05 complete. Next: Notebook 06 — Full Benchmark.")

---
## 16. Conclusions

| Metric | RF (NB03) | XGBoost (NB04) | LSTM (NB05) |
|---|---|---|---|
| Accuracy | 99.94% | 99.94% | — |
| F1 (macro) | 99.94% | 99.94% | — |
| Recall | 99.96% | 99.98% | — |
| ROC-AUC | 1.0000 | 1.0000 | — |
| FPR | 0.09% | 0.09% | — |
| Train time | 220 min | 84.6 min | — |

> Fill in LSTM column after running the notebook.

### Next steps
- [ ] **Notebook 06** — Full benchmark: RF vs XGBoost vs LSTM vs Snort vs Hybrid
- [ ] **Notebook 07** — IDS Testing: tcpreplay, hping3, nmap